In [ ]:
import sys
import os
from pathlib import Path

# Add project root to Python path to access local segment-anything
project_root = Path('.').resolve().parent
sys.path.insert(0, str(project_root))

print(f"📁 Project root: {project_root}")
print(f"🔍 Looking for segment-anything at: {project_root / 'segment-anything'}")

# Verify local segment-anything exists
sam_path = project_root / 'segment-anything'
if sam_path.exists():
    print(f"✅ Found local segment-anything directory")
    # Add segment-anything to path for direct imports
    sys.path.insert(0, str(sam_path))
else:
    print(f"❌ Local segment-anything directory not found!")
    raise FileNotFoundError(f"segment-anything directory not found at {sam_path}")

# Import SAM modules from LOCAL source code
print("📦 Importing SAM modules from local source...")
try:
    from segment_anything import (
        build_sam, 
        build_sam_vit_h, 
        build_sam_vit_l, 
        build_sam_vit_b,
        sam_model_registry,
        SamPredictor,
        SamAutomaticMaskGenerator
    )
    print("✅ Successfully imported SAM modules from local source!")
    print(f"📍 SAM module location: {build_sam.__module__}")
except ImportError as e:
    print(f"❌ Failed to import SAM modules: {e}")
    raise

# Import other required libraries
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import nibabel as nib
from tqdm import tqdm
import cv2
from scipy import ndimage
from sklearn.metrics import jaccard_score
import warnings
warnings.filterwarnings("ignore")

print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🖥️  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

print("\n🎯 SAM Baseline Test Environment Ready!")
print("💡 Any changes to segment-anything/ source code will be immediately available!")


In [ ]:
# SAM model checkpoints - Update these paths to your downloaded models
SAM_CHECKPOINT_PATHS = {
    "vit_b": "../pretrained/sam_vit_b_01ec64.pth",
    "vit_l": "../pretrained/sam_vit_l_0b3195.pth", 
    "vit_h": "../pretrained/sam_vit_h_4b8939.pth"
}

def load_sam_model(model_type="vit_b", device="cuda" if torch.cuda.is_available() else "cpu"):
    """
    Load SAM model using LOCAL source code.
    
    Args:
        model_type (str): Model variant ("vit_b", "vit_l", "vit_h")
        device (str): Device to load model on
    
    Returns:
        SAM model and predictor
    """
    print(f"🤖 Loading SAM model: {model_type}")
    print(f"📍 Using local SAM source from: {build_sam.__module__}")
    
    checkpoint_path = SAM_CHECKPOINT_PATHS.get(model_type)
    
    if not checkpoint_path or not Path(checkpoint_path).exists():
        print(f"⚠️  Checkpoint not found: {checkpoint_path}")
        print("💡 Available options:")
        for key, path in SAM_CHECKPOINT_PATHS.items():
            exists = "✅" if Path(path).exists() else "❌"
            print(f"   {exists} {key}: {path}")
        
        # Try to find any available checkpoint
        for key, path in SAM_CHECKPOINT_PATHS.items():
            if Path(path).exists():
                print(f"🔄 Using available checkpoint: {key}")
                model_type = key
                checkpoint_path = path
                break
        else:
            print("❌ No SAM checkpoints found!")
            print("💡 Download SAM checkpoints from:")
            print("   https://github.com/facebookresearch/segment-anything#model-checkpoints")
            return None, None
    
    try:
        # Load model using local source code
        if model_type == "vit_b":
            sam = build_sam_vit_b(checkpoint=checkpoint_path)
        elif model_type == "vit_l":
            sam = build_sam_vit_l(checkpoint=checkpoint_path)
        elif model_type == "vit_h":
            sam = build_sam_vit_h(checkpoint=checkpoint_path)
        else:
            # Generic loader
            sam = build_sam(checkpoint=checkpoint_path)
        
        sam.to(device)
        sam.eval()
        
        # Create predictor using local source
        predictor = SamPredictor(sam)
        
        print(f"✅ SAM model loaded successfully!")
        print(f"🔧 Model type: {model_type}")
        print(f"🖥️  Device: {device}")
        print(f"⚙️  Parameters: {sum(p.numel() for p in sam.parameters()):,}")
        
        # Display model architecture info
        print(f"\n🏗️  Model Architecture (from local source):")
        print(f"   Image Encoder: {type(sam.image_encoder).__name__}")
        print(f"   Prompt Encoder: {type(sam.prompt_encoder).__name__}")
        print(f"   Mask Decoder: {type(sam.mask_decoder).__name__}")
        
        return sam, predictor
        
    except Exception as e:
        print(f"❌ Failed to load SAM model: {e}")
        return None, None

# Load SAM model using local source code
sam_model, sam_predictor = load_sam_model("vit_b")

if sam_model is not None:
    print(f"\n🎯 SAM Model Ready for Medical Image Segmentation!")
    print(f"💡 Source files can be modified in: {sam_path}/segment_anything/")
else:
    print(f"❌ SAM model loading failed. Please check checkpoint paths.")


In [ ]:
def dice_score(pred, target, smooth=1e-6):
    """Calculate Dice Similarity Coefficient."""
    pred_flat = pred.flatten()
    target_flat = target.flatten()
    intersection = (pred_flat * target_flat).sum()
    return (2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

def iou_score(pred, target, smooth=1e-6):
    """Calculate Intersection over Union."""
    pred_flat = pred.flatten()
    target_flat = target.flatten()
    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    return (intersection + smooth) / (union + smooth)

def hausdorff_distance(pred, target):
    """Calculate Hausdorff Distance."""
    try:
        from scipy.spatial.distance import directed_hausdorff
        pred_coords = np.column_stack(np.where(pred > 0))
        target_coords = np.column_stack(np.where(target > 0))
        
        if len(pred_coords) == 0 or len(target_coords) == 0:
            return float('inf')
        
        hd1 = directed_hausdorff(pred_coords, target_coords)[0]
        hd2 = directed_hausdorff(target_coords, pred_coords)[0]
        return max(hd1, hd2)
    except:
        return float('inf')

def sensitivity(pred, target, smooth=1e-6):
    """Calculate Sensitivity (True Positive Rate)."""
    tp = (pred * target).sum()
    fn = (target * (1 - pred)).sum()
    return (tp + smooth) / (tp + fn + smooth)

def specificity(pred, target, smooth=1e-6):
    """Calculate Specificity (True Negative Rate)."""
    tn = ((1 - pred) * (1 - target)).sum()
    fp = (pred * (1 - target)).sum()
    return (tn + smooth) / (tn + fp + smooth)

def calculate_metrics(pred_mask, gt_mask):
    """Calculate all evaluation metrics."""
    # Ensure binary masks
    pred_binary = (pred_mask > 0.5).astype(np.float32)
    gt_binary = (gt_mask > 0.5).astype(np.float32)
    
    metrics = {
        'dice': dice_score(pred_binary, gt_binary),
        'iou': iou_score(pred_binary, gt_binary),
        'hausdorff': hausdorff_distance(pred_binary, gt_binary),
        'sensitivity': sensitivity(pred_binary, gt_binary),
        'specificity': specificity(pred_binary, gt_binary)
    }
    return metrics

print("📊 Evaluation metrics ready!")


In [ ]:
def sam_segment(image, bbox, predictor=None, debug=False):
    """
    Segment image using SAM with bounding box prompt.
    Uses LOCAL SAM source code for full customization capability.
    
    Args:
        image (np.ndarray): Input image (H, W, 3) in RGB format
        bbox (list): Bounding box [x1, y1, x2, y2]
        predictor (SamPredictor): SAM predictor from local source
        debug (bool): Enable debug output
    
    Returns:
        np.ndarray: Binary segmentation mask
    """
    if predictor is None:
        if debug:
            print("❌ SAM predictor not provided")
        return np.zeros(image.shape[:2], dtype=np.uint8)
    
    try:
        if debug:
            print(f"🎯 SAM segmentation using LOCAL source code")
            print(f"   Image shape: {image.shape}")
            print(f"   Bbox: {bbox}")
            print(f"   Predictor type: {type(predictor)}")
            print(f"   Predictor module: {predictor.__module__}")
        
        # Set image for SAM predictor (from local source)
        predictor.set_image(image)
        
        # Convert bbox to SAM format
        input_box = np.array(bbox)
        
        # Predict masks using local SAM predictor
        masks, scores, logits = predictor.predict(
            point_coords=None,
            point_labels=None,
            box=input_box[None, :],
            multimask_output=True,
        )
        
        # Select best mask (highest score)
        best_mask_idx = np.argmax(scores)
        best_mask = masks[best_mask_idx]
        
        if debug:
            print(f"   Generated {len(masks)} masks")
            print(f"   Best mask score: {scores[best_mask_idx]:.3f}")
            print(f"   Mask shape: {best_mask.shape}")
            print(f"   Mask coverage: {best_mask.sum()/best_mask.size:.3f}")
        
        return best_mask.astype(np.uint8)
        
    except Exception as e:
        if debug:
            print(f"❌ SAM segmentation failed: {e}")
        return np.zeros(image.shape[:2], dtype=np.uint8)

def get_bbox_from_mask(mask, padding=10):
    """Extract bounding box from ground truth mask."""
    coords = np.column_stack(np.where(mask > 0))
    if len(coords) == 0:
        return [0, 0, mask.shape[1], mask.shape[0]]
    
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    
    # Add padding
    y_min = max(0, y_min - padding)
    x_min = max(0, x_min - padding)
    y_max = min(mask.shape[0], y_max + padding)
    x_max = min(mask.shape[1], x_max + padding)
    
    return [x_min, y_min, x_max, y_max]

def normalize_image_for_sam(image):
    """Normalize medical image for SAM input."""
    # Convert to RGB if grayscale
    if len(image.shape) == 2:
        image = np.stack([image] * 3, axis=-1)
    elif image.shape[2] == 1:
        image = np.repeat(image, 3, axis=2)
    
    # Normalize to [0, 255] range
    if image.max() <= 1.0:
        image = (image * 255).astype(np.uint8)
    else:
        image = ((image - image.min()) / (image.max() - image.min()) * 255).astype(np.uint8)
    
    return image

# Test the SAM segmentation function
if sam_predictor is not None:
    print("🧪 Testing SAM segmentation function with local source code...")
    
    # Create test image and bbox
    test_image = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
    test_bbox = [50, 50, 150, 150]
    
    # Test segmentation
    test_mask = sam_segment(test_image, test_bbox, sam_predictor, debug=True)
    print(f"✅ SAM function test completed. Mask shape: {test_mask.shape}")
    
    print(f"\n💡 You can now modify the SAM source code in:")
    print(f"   📁 {sam_path}/segment_anything/predictor.py")
    print(f"   📁 {sam_path}/segment_anything/modeling/")
    print(f"   📁 {sam_path}/segment_anything/build_sam.py")
    print(f"   Changes will be immediately reflected in this notebook!")
else:
    print("⚠️  SAM predictor not available. Skipping function test.")


In [ ]:
def create_synthetic_medical_data():
    """Create synthetic medical imaging data for SAM testing."""
    datasets = {}
    
    # 1. MRI Brain Hippocampus - High contrast structure
    print("🧠 Creating synthetic MRI Brain Hippocampus dataset...")
    mri_images = []
    mri_masks = []
    
    for i in range(5):
        # Create brain-like structure
        img = np.zeros((256, 256))
        y, x = np.ogrid[:256, :256]
        center_y, center_x = 128 + np.random.randint(-20, 20), 128 + np.random.randint(-20, 20)
        
        # Brain outline
        brain_mask = (x - center_x)**2 + (y - center_y)**2 < 100**2
        img[brain_mask] = 0.8 + np.random.normal(0, 0.1, brain_mask.sum())
        
        # Hippocampus-like structure
        hippo_y, hippo_x = center_y + 30, center_x - 40
        hippocampus = ((x - hippo_x)**2/15**2 + (y - hippo_y)**2/8**2 < 1) & brain_mask
        img[hippocampus] = 1.2 + np.random.normal(0, 0.05, hippocampus.sum())
        
        # Add noise
        img += np.random.normal(0, 0.05, img.shape)
        img = np.clip(img, 0, 2)
        
        mri_images.append(img)
        mri_masks.append(hippocampus.astype(np.uint8))
    
    datasets['mri_hippocampus'] = {
        'images': mri_images,
        'masks': mri_masks,
        'modality': 'MRI',
        'organ': 'Brain Hippocampus',
        'description': 'High contrast subcortical structure'
    }
    
    # 2. CT Spleen - Abdominal organ
    print("🫁 Creating synthetic CT Spleen dataset...")
    ct_images = []
    ct_masks = []
    
    for i in range(5):
        # Create abdominal CT-like image
        img = np.random.normal(-1000, 100, (256, 256))  # Air/background
        
        # Body outline
        body_y, body_x = np.ogrid[:256, :256]
        body_center_y, body_center_x = 128, 128
        body_mask = (body_x - body_center_x)**2 + (body_y - body_center_y)**2 < 110**2
        img[body_mask] = np.random.normal(-50, 50, body_mask.sum())  # Soft tissue
        
        # Spleen-like structure
        spleen_y, spleen_x = body_center_y - 30, body_center_x - 60
        spleen_mask = ((body_x - spleen_x)**2/20**2 + (body_y - spleen_y)**2/30**2 < 1) & body_mask
        img[spleen_mask] = np.random.normal(50, 20, spleen_mask.sum())  # Spleen density
        
        # Normalize to 0-1 range
        img = (img + 1000) / 2000
        img = np.clip(img, 0, 1)
        
        ct_images.append(img)
        ct_masks.append(spleen_mask.astype(np.uint8))
    
    datasets['ct_spleen'] = {
        'images': ct_images,
        'masks': ct_masks,
        'modality': 'CT',
        'organ': 'Spleen',
        'description': 'Abdominal organ with moderate contrast'
    }
    
    # 3. Ultrasound Breast - Low contrast, noisy
    print("🔍 Creating synthetic Ultrasound Breast dataset...")
    us_images = []
    us_masks = []
    
    for i in range(5):
        # Create ultrasound-like image with noise
        img = np.random.exponential(0.3, (256, 256))
        
        # Add ultrasound-specific noise patterns
        for _ in range(20):
            y_noise = np.random.randint(0, 256)
            x_noise = np.random.randint(0, 256)
            noise_size = np.random.randint(5, 15)
            y_slice = slice(max(0, y_noise-noise_size), min(256, y_noise+noise_size))
            x_slice = slice(max(0, x_noise-noise_size), min(256, x_noise+noise_size))
            img[y_slice, x_slice] *= np.random.uniform(0.5, 2.0)
        
        # Breast lesion/mass
        lesion_y, lesion_x = 128 + np.random.randint(-40, 40), 128 + np.random.randint(-40, 40)
        lesion_size = np.random.randint(15, 25)
        y, x = np.ogrid[:256, :256]
        lesion_mask = (x - lesion_x)**2 + (y - lesion_y)**2 < lesion_size**2
        img[lesion_mask] = np.random.normal(0.7, 0.1, lesion_mask.sum())
        
        # Normalize
        img = np.clip(img, 0, 1)
        
        us_images.append(img)
        us_masks.append(lesion_mask.astype(np.uint8))
    
    datasets['ultrasound_breast'] = {
        'images': us_images,
        'masks': us_masks,
        'modality': 'Ultrasound',
        'organ': 'Breast',
        'description': 'Low contrast with high noise'
    }
    
    return datasets

# Load synthetic datasets
print("📊 Creating synthetic medical datasets for SAM evaluation...")
datasets = create_synthetic_medical_data()

# Display dataset information
print(f"\n📋 Dataset Summary:")
for name, data in datasets.items():
    print(f"  🏥 {name}: {len(data['images'])} images")
    print(f"     📸 Modality: {data['modality']}")
    print(f"     🫀 Organ: {data['organ']}")
    print(f"     📝 Description: {data['description']}")
    print(f"     📏 Image shape: {data['images'][0].shape}")
    print(f"     🎯 Mask stats: {data['masks'][0].sum()} pixels")
    print()

print("✅ Synthetic datasets ready for SAM evaluation!")


In [ ]:
def evaluate_sam_on_dataset(dataset_name, dataset_data, predictor, verbose=True):
    """
    Evaluate SAM on a specific dataset using LOCAL source code.
    
    Args:
        dataset_name (str): Name of the dataset
        dataset_data (dict): Dataset containing images and masks
        predictor (SamPredictor): SAM predictor from local source
        verbose (bool): Print progress information
    
    Returns:
        dict: Evaluation results
    """
    if predictor is None:
        print(f"❌ No SAM predictor available for {dataset_name}")
        return None
    
    images = dataset_data['images']
    masks = dataset_data['masks']
    modality = dataset_data['modality']
    organ = dataset_data['organ']
    
    if verbose:
        print(f"\n🔬 Evaluating SAM on {dataset_name}")
        print(f"   📸 Modality: {modality}")
        print(f"   🫀 Organ: {organ}")
        print(f"   📊 Images: {len(images)}")
        print(f"   🎯 Using LOCAL SAM source code")
    
    results = []
    
    for i, (image, gt_mask) in enumerate(zip(images, masks)):
        if verbose:
            print(f"   Processing image {i+1}/{len(images)}...", end="")
            
        try:
            # Normalize image for SAM
            sam_image = normalize_image_for_sam(image)
            
            # Get bounding box from ground truth mask
            bbox = get_bbox_from_mask(gt_mask, padding=10)
            
            # Run SAM segmentation using local source
            pred_mask = sam_segment(sam_image, bbox, predictor, debug=False)
            
            # Calculate metrics
            metrics = calculate_metrics(pred_mask, gt_mask)
            metrics['image_id'] = i
            metrics['bbox'] = bbox
            results.append(metrics)
            
            if verbose:
                print(f" Dice: {metrics['dice']:.3f}")
                
        except Exception as e:
            if verbose:
                print(f" ❌ Error: {str(e)[:50]}...")
            # Add failed result
            results.append({
                'image_id': i,
                'dice': 0.0,
                'iou': 0.0,
                'hausdorff': float('inf'),
                'sensitivity': 0.0,
                'specificity': 0.0,
                'bbox': [0, 0, 0, 0]
            })
    
    # Calculate summary statistics
    summary = {
        'dataset': dataset_name,
        'modality': modality,
        'organ': organ,
        'num_images': len(images),
        'results': results
    }
    
    # Calculate averages (excluding failed cases)
    valid_results = [r for r in results if r['dice'] > 0]
    if valid_results:
        summary['avg_dice'] = np.mean([r['dice'] for r in valid_results])
        summary['avg_iou'] = np.mean([r['iou'] for r in valid_results])
        summary['avg_hausdorff'] = np.mean([r['hausdorff'] for r in valid_results if r['hausdorff'] != float('inf')])
        summary['avg_sensitivity'] = np.mean([r['sensitivity'] for r in valid_results])
        summary['avg_specificity'] = np.mean([r['specificity'] for r in valid_results])
        summary['success_rate'] = len(valid_results) / len(results)
    else:
        summary['avg_dice'] = 0.0
        summary['avg_iou'] = 0.0
        summary['avg_hausdorff'] = float('inf')
        summary['avg_sensitivity'] = 0.0
        summary['avg_specificity'] = 0.0
        summary['success_rate'] = 0.0
    
    if verbose:
        print(f"   ✅ Completed! Average Dice: {summary['avg_dice']:.3f}")
    
    return summary

# Main evaluation loop
print("🚀 Starting SAM evaluation using LOCAL source code...")
print(f"💡 SAM predictor module: {sam_predictor.__module__ if sam_predictor else 'None'}")

all_results = {}

if sam_predictor is not None:
    for dataset_name, dataset_data in datasets.items():
        result = evaluate_sam_on_dataset(dataset_name, dataset_data, sam_predictor, verbose=True)
        if result is not None:
            all_results[dataset_name] = result
    
    print(f"\n📊 SAM Evaluation Complete!")
    print(f"   📁 Datasets evaluated: {len(all_results)}")
    print(f"   🎯 Using LOCAL SAM source from: {sam_path}")
    
else:
    print("❌ SAM predictor not available. Cannot run evaluation.")
    print("💡 Please ensure SAM model is loaded correctly.")


In [ ]:
def visualize_sam_results(results, datasets, max_examples=2):
    """Visualize SAM segmentation results from local source code."""
    
    if not results:
        print("❌ No results to visualize")
        return
    
    print(f"🖼️  Visualizing SAM results (using local source code)")
    print(f"📍 SAM source location: {sam_path}")
    
    for dataset_name, result in results.items():
        dataset_data = datasets[dataset_name]
        images = dataset_data['images']
        masks = dataset_data['masks']
        
        print(f"\n📊 {dataset_name} - {result['modality']} {result['organ']}")
        print(f"   Average Dice: {result['avg_dice']:.3f}")
        print(f"   Average IoU: {result['avg_iou']:.3f}")
        
        # Show first few examples
        num_examples = min(max_examples, len(images))
        fig, axes = plt.subplots(num_examples, 4, figsize=(16, 4*num_examples))
        if num_examples == 1:
            axes = axes.reshape(1, -1)
        
        for i in range(num_examples):
            image = images[i]
            gt_mask = masks[i]
            
            # Get prediction by re-running SAM
            sam_image = normalize_image_for_sam(image)
            bbox = get_bbox_from_mask(gt_mask, padding=10)
            
            if sam_predictor is not None:
                pred_mask = sam_segment(sam_image, bbox, sam_predictor, debug=False)
            else:
                pred_mask = np.zeros_like(gt_mask)
            
            metrics = result['results'][i]
            
            # Original image
            axes[i, 0].imshow(image, cmap='gray')
            axes[i, 0].set_title(f'Original Image {i+1}')
            axes[i, 0].axis('off')
            
            # Ground truth
            axes[i, 1].imshow(image, cmap='gray', alpha=0.7)
            axes[i, 1].imshow(gt_mask, cmap='Reds', alpha=0.5)
            axes[i, 1].set_title('Ground Truth')
            axes[i, 1].axis('off')
            
            # SAM prediction
            axes[i, 2].imshow(image, cmap='gray', alpha=0.7)
            axes[i, 2].imshow(pred_mask, cmap='Blues', alpha=0.5)
            axes[i, 2].set_title(f'SAM Prediction')
            axes[i, 2].axis('off')
            
            # Overlay comparison
            axes[i, 3].imshow(image, cmap='gray', alpha=0.7)
            axes[i, 3].imshow(gt_mask, cmap='Reds', alpha=0.3, label='GT')
            axes[i, 3].imshow(pred_mask, cmap='Blues', alpha=0.3, label='SAM')
            axes[i, 3].set_title(f'Dice: {metrics["dice"]:.3f}')
            axes[i, 3].axis('off')
            
            # Add bounding box
            bbox = metrics['bbox']
            from matplotlib.patches import Rectangle
            rect = Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1], 
                           linewidth=2, edgecolor='yellow', facecolor='none')
            axes[i, 3].add_patch(rect)
        
        plt.tight_layout()
        plt.show()

# Visualize results
if all_results:
    visualize_sam_results(all_results, datasets, max_examples=2)
else:
    print("❌ No results to visualize. Please run the evaluation first.")
    
    # Show demo visualization instead
    print("📸 Creating demo visualization...")
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    # Create demo data
    demo_image = np.random.rand(128, 128)
    demo_gt = np.zeros((128, 128))
    demo_gt[40:80, 40:80] = 1
    demo_pred = np.zeros((128, 128))
    demo_pred[45:85, 35:75] = 1
    
    axes[0].imshow(demo_image, cmap='gray')
    axes[0].set_title('Demo Image')
    axes[0].axis('off')
    
    axes[1].imshow(demo_image, cmap='gray', alpha=0.7)
    axes[1].imshow(demo_gt, cmap='Reds', alpha=0.5)
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')
    
    axes[2].imshow(demo_image, cmap='gray', alpha=0.7)
    axes[2].imshow(demo_pred, cmap='Blues', alpha=0.5)
    axes[2].set_title('SAM Prediction')
    axes[2].axis('off')
    
    axes[3].imshow(demo_image, cmap='gray', alpha=0.7)
    axes[3].imshow(demo_gt, cmap='Reds', alpha=0.3)
    axes[3].imshow(demo_pred, cmap='Blues', alpha=0.3)
    axes[3].set_title('Overlay')
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("💡 This is a demo visualization. Load SAM model to see real results.")


In [ ]:
def print_detailed_results(results):
    """Print detailed evaluation results."""
    if not results:
        print("❌ No results available")
        return
    
    print("="*60)
    print("🎯 SAM BASELINE TEST RESULTS (Using Local Source Code)")
    print("="*60)
    print(f"📍 SAM Source Location: {sam_path}")
    print(f"🔧 SAM Predictor Module: {sam_predictor.__module__ if sam_predictor else 'Not loaded'}")
    print()
    
    # Summary table
    print("📊 PERFORMANCE SUMMARY")
    print("-"*60)
    print(f"{'Dataset':<20} {'Modality':<12} {'Dice':<8} {'IoU':<8} {'Sens':<8} {'Spec':<8}")
    print("-"*60)
    
    overall_dice = []
    overall_iou = []
    
    for dataset_name, result in results.items():
        overall_dice.append(result['avg_dice'])
        overall_iou.append(result['avg_iou'])
        
        print(f"{dataset_name:<20} {result['modality']:<12} "
              f"{result['avg_dice']:<8.3f} {result['avg_iou']:<8.3f} "
              f"{result['avg_sensitivity']:<8.3f} {result['avg_specificity']:<8.3f}")
    
    print("-"*60)
    if overall_dice:
        print(f"{'OVERALL':<20} {'ALL':<12} "
              f"{np.mean(overall_dice):<8.3f} {np.mean(overall_iou):<8.3f} "
              f"{'N/A':<8} {'N/A':<8}")
    print()
    
    # Detailed analysis
    print("📋 DETAILED ANALYSIS")
    print("-"*60)
    
    for dataset_name, result in results.items():
        print(f"\n🏥 {dataset_name.upper()}")
        print(f"   📸 Modality: {result['modality']}")
        print(f"   🫀 Organ: {result['organ']}")
        print(f"   📊 Images: {result['num_images']}")
        print(f"   ✅ Success Rate: {result['success_rate']:.1%}")
        print(f"   📈 Metrics:")
        print(f"      • Dice Score: {result['avg_dice']:.3f}")
        print(f"      • IoU Score: {result['avg_iou']:.3f}")
        if result['avg_hausdorff'] != float('inf'):
            print(f"      • Hausdorff Distance: {result['avg_hausdorff']:.2f}")
        print(f"      • Sensitivity: {result['avg_sensitivity']:.3f}")
        print(f"      • Specificity: {result['avg_specificity']:.3f}")
    
    print()
    print("🔍 SAM ANALYSIS FOR MEDICAL IMAGING")
    print("-"*60)
    
    if overall_dice:
        avg_performance = np.mean(overall_dice)
        if avg_performance > 0.8:
            performance_level = "Excellent"
        elif avg_performance > 0.6:
            performance_level = "Good"
        elif avg_performance > 0.4:
            performance_level = "Moderate"
        else:
            performance_level = "Poor"
        
        print(f"📊 Overall Performance: {performance_level} (Dice: {avg_performance:.3f})")
        
        # Performance by modality
        modality_performance = {}
        for result in results.values():
            modality = result['modality']
            if modality not in modality_performance:
                modality_performance[modality] = []
            modality_performance[modality].append(result['avg_dice'])
        
        print(f"📈 Performance by Modality:")
        for modality, scores in modality_performance.items():
            print(f"   • {modality}: {np.mean(scores):.3f} ± {np.std(scores):.3f}")
    
    print()
    print("💡 SAM CHARACTERISTICS:")
    print("   • General-purpose model trained on natural images")
    print("   • Requires prompts (bounding boxes) for segmentation")
    print("   • No medical domain-specific training")
    print("   • Performance varies by image contrast and structure complexity")
    print()
    print("🔧 LOCAL SOURCE CODE BENEFITS:")
    print("   • Full access to model architecture and implementation")
    print("   • Ability to modify and debug any component")
    print("   • Immediate reflection of code changes")
    print("   • Complete transparency for research and development")

# Print results
if all_results:
    print_detailed_results(all_results)
else:
    print("❌ No evaluation results available.")
    print("💡 Please ensure:")
    print("   1. SAM model checkpoints are downloaded")
    print("   2. Model loading completed successfully")
    print("   3. Evaluation was run without errors")
    
    print(f"\n🎯 SAM Local Source Code Setup:")
    print(f"   📁 Source path: {sam_path}")
    print(f"   ✅ Directory exists: {sam_path.exists()}")
    if sam_path.exists():
        key_files = [
            "segment_anything/__init__.py",
            "segment_anything/predictor.py", 
            "segment_anything/build_sam.py",
            "segment_anything/modeling/"
        ]
        print(f"   📋 Key files:")
        for file in key_files:
            file_path = sam_path / file
            exists = "✅" if file_path.exists() else "❌"
            print(f"      {exists} {file}")
    
    print(f"\n💡 You can now modify SAM source code and see immediate effects!")
    print(f"   🔧 Edit predictor logic: {sam_path}/segment_anything/predictor.py")
    print(f"   🏗️  Modify architecture: {sam_path}/segment_anything/modeling/")
    print(f"   ⚙️  Change build process: {sam_path}/segment_anything/build_sam.py")


In [ ]:
print("🔧 SAM LOCAL SOURCE CODE MODIFICATION GUIDE")
print("="*60)
print(f"📁 Your SAM source code is located at: {sam_path}")
print()

print("📋 KEY FILES YOU CAN MODIFY:")
print("-"*40)

key_files = {
    "segment_anything/predictor.py": "Main prediction interface - modify prompting logic",
    "segment_anything/build_sam.py": "Model building - add custom architectures", 
    "segment_anything/modeling/sam.py": "Core SAM model - modify architecture",
    "segment_anything/modeling/image_encoder.py": "Vision Transformer encoder",
    "segment_anything/modeling/mask_decoder.py": "Mask prediction head",
    "segment_anything/modeling/prompt_encoder.py": "Prompt encoding (boxes, points)",
    "segment_anything/automatic_mask_generator.py": "Automatic mask generation",
    "segment_anything/utils/transforms.py": "Image preprocessing transforms"
}

for file_path, description in key_files.items():
    full_path = sam_path / file_path
    exists = "✅" if full_path.exists() else "❌"
    print(f"{exists} {file_path}")
    print(f"     💡 {description}")
    print()

print("🛠️  COMMON MODIFICATIONS FOR MEDICAL IMAGING:")
print("-"*50)

modifications = [
    ("Medical-specific preprocessing", 
     "segment_anything/utils/transforms.py",
     "Add DICOM handling, window/level adjustments, medical normalization"),
    
    ("Custom prompt strategies", 
     "segment_anything/predictor.py", 
     "Implement anatomical priors, multi-scale prompting"),
    
    ("Architecture adaptations",
     "segment_anything/modeling/image_encoder.py",
     "Add domain adaptation layers, modify attention mechanisms"),
    
    ("Loss function modifications",
     "segment_anything/modeling/mask_decoder.py", 
     "Add medical-specific losses (boundary, topology preservation)"),
    
    ("Multi-class segmentation",
     "segment_anything/modeling/sam.py",
     "Extend for simultaneous multi-organ segmentation"),
    
    ("3D medical image support",
     "segment_anything/modeling/",
     "Adapt 2D components for 3D medical volumes")
]

for i, (task, file_path, description) in enumerate(modifications, 1):
    print(f"{i}. {task}")
    print(f"   📁 File: {file_path}")
    print(f"   💡 {description}")
    print()

print("🚀 EXAMPLE MODIFICATION WORKFLOW:")
print("-"*40)
print("1. 📝 Edit the source file (e.g., segment_anything/predictor.py)")
print("2. 💾 Save your changes")
print("3. 🔄 Restart this notebook kernel (Kernel → Restart)")
print("4. ▶️  Re-run the notebook - your changes will be active!")
print("5. 🧪 Test your modifications with the evaluation code")
print()

print("💡 DEBUGGING TIPS:")
print("-"*20)
print("• Add print statements in the SAM source code to trace execution")
print("• Use `sam_segment(..., debug=True)` to enable verbose output")
print("• Check `sam_predictor.__module__` to confirm local import")
print("• Modify `build_sam.py` to add custom model variants")
print()

print("🔍 EXAMPLE: Adding Debug Prints to SAM Predictor")
print("-"*50)
predictor_file = sam_path / "segment_anything" / "predictor.py"
if predictor_file.exists():
    print(f"✅ You can edit: {predictor_file}")
    print()
    print("Example modification in predictor.py:")
    print("```python")
    print("def predict(self, ...):")
    print("    print(f'SAM DEBUG: Processing image shape {self._image_shape}')")
    print("    print(f'SAM DEBUG: Input prompts - boxes: {box}, points: {point_coords}')")
    print("    # ... existing code ...")
    print("    print(f'SAM DEBUG: Generated {len(masks)} masks with scores {scores}')")
    print("    return masks, scores, logits")
    print("```")
    print()
    print("After editing, restart the kernel and re-run - you'll see debug output!")
else:
    print("❌ Predictor file not found")

print()
print("🎯 NEXT STEPS:")
print("1. Choose a modification from the list above")
print("2. Edit the corresponding file in your local segment-anything directory")  
print("3. Restart this notebook and re-run to test your changes")
print("4. Compare results before and after your modifications")
print()
print("✨ Happy coding! Your SAM modifications are now immediately usable! ✨")


In [ ]:
# Environment setup and Colab detection
# Ref: setup_colab.ipynb for Colab integration patterns
import sys
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🌐 Running in Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')

    # Set up paths for Colab - following setup_colab.ipynb patterns
    DRIVE_ROOT = '/content/drive/MyDrive'
    PROJECT_ROOT = f'{DRIVE_ROOT}/RL-CC-SAM'

    # Change to project directory
    os.chdir(PROJECT_ROOT)
    sys.path.append(PROJECT_ROOT)

    print(f"📁 Working directory: {os.getcwd()}")
else:
    print("💻 Running locally")
    # Assume notebook is in notebooks/ folder
    PROJECT_ROOT = Path.cwd().parent
    os.chdir(PROJECT_ROOT)

    print(f"📁 Working directory: {PROJECT_ROOT}")

# Set up directories following project structure
DATASETS_DIR = Path(PROJECT_ROOT) / "datasets"
PRETRAINED_DIR = Path(PROJECT_ROOT) / "pretrained"
SEGMENT_ANYTHING_DIR = Path(PROJECT_ROOT) / "segment-anything"

# Create directories
for dir_path in [DATASETS_DIR, PRETRAINED_DIR]:
    dir_path.mkdir(exist_ok=True)

print(f"📊 Datasets directory: {DATASETS_DIR}")
print(f"🧠 Pretrained models directory: {PRETRAINED_DIR}")
print(f"🔧 Segment Anything submodule: {SEGMENT_ANYTHING_DIR}")

# Verify submodules are available (following .cursor/rules/submodule-handling.mdc)
if not SEGMENT_ANYTHING_DIR.exists():
    print("❌ Segment Anything submodule not found!")
    print("💡 Run: git submodule update --init --recursive")
    print("💡 Or use setup_colab.ipynb for initial setup")
else:
    print("✅ Segment Anything submodule found")

print("✅ Environment setup complete")


In [ ]:
# Import essential libraries
# All dependencies should be installed via requirements.txt

# Core libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import json
import math
import glob
import sys
from tqdm.auto import tqdm

# Medical imaging
import nibabel as nib
from scipy.ndimage import distance_transform_edt

# SAM - using existing submodule (following .cursor/rules/submodule-handling.mdc)
sys.path.insert(0, str(SEGMENT_ANYTHING_DIR))
from segment_anything import sam_model_registry, SamPredictor

# Image processing
from skimage import io, transform
import torch.nn.functional as F

# Device handling following .cursor/rules/pytorch-devices.mdc
def get_device():
    """Get optimal device following PyTorch device handling rules."""
    if torch.backends.mps.is_available():
        return torch.device("mps")  # Apple Silicon
    elif torch.cuda.is_available():
        return torch.device("cuda")  # NVIDIA GPU
    else:
        return torch.device("cpu")   # CPU fallback

device = get_device()
print(f"🚀 Using device: {device}")

# Set style for plots
plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
print("✅ Libraries imported successfully")


In [ ]:
# Define comprehensive evaluation metrics for medical image segmentation
def compute_dice(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """
    Calculate Dice Coefficient (F1 score) between two binary masks.

    Args:
        pred_mask: Predicted binary mask
        true_mask: Ground truth binary mask

    Returns:
        float: Dice coefficient [0, 1], where 1 is perfect overlap
    """
    intersection = np.logical_and(pred_mask, true_mask).sum()
    size_pred = pred_mask.sum()
    size_true = true_mask.sum()

    if size_pred + size_true == 0:
        return 1.0  # Both masks are empty - perfect agreement

    return 2.0 * intersection / (size_pred + size_true + 1e-8)

def compute_iou(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """
    Calculate Intersection over Union (IoU) between two binary masks.

    Args:
        pred_mask: Predicted binary mask
        true_mask: Ground truth binary mask

    Returns:
        float: IoU score [0, 1], where 1 is perfect overlap
    """
    intersection = np.logical_and(pred_mask, true_mask).sum()
    union = np.logical_or(pred_mask, true_mask).sum()

    if union == 0:
        return 1.0  # Both masks are empty

    return intersection / (union + 1e-8)

def compute_hausdorff(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """
    Calculate Hausdorff distance between two binary masks.

    Args:
        pred_mask: Predicted binary mask
        true_mask: Ground truth binary mask

    Returns:
        float: Hausdorff distance in pixels (lower is better)
    """
    if pred_mask.sum() == 0 or true_mask.sum() == 0:
        return math.inf  # Cannot compute distance if either mask is empty

    # Calculate distance transforms
    dt_true = distance_transform_edt(~true_mask.astype(bool))
    dt_pred = distance_transform_edt(~pred_mask.astype(bool))

    # Calculate directed Hausdorff distances
    hd1 = dt_pred[true_mask.astype(bool)].max()  # GT boundary to pred region
    hd2 = dt_true[pred_mask.astype(bool)].max()  # Pred boundary to GT region

    return max(hd1, hd2)

def compute_sensitivity(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """Calculate sensitivity (recall/true positive rate)."""
    tp = np.logical_and(pred_mask, true_mask).sum()
    fn = np.logical_and(~pred_mask, true_mask).sum()

    if tp + fn == 0:
        return 1.0  # No positive cases

    return tp / (tp + fn)

def compute_specificity(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """Calculate specificity (true negative rate)."""
    tn = np.logical_and(~pred_mask, ~true_mask).sum()
    fp = np.logical_and(pred_mask, ~true_mask).sum()

    if tn + fp == 0:
        return 1.0  # No negative cases

    return tn / (tn + fp)

def evaluate_segmentation(pred_mask: np.ndarray, true_mask: np.ndarray) -> dict:
    """
    Comprehensive evaluation of segmentation performance.

    Returns:
        dict: Dictionary containing all evaluation metrics
    """
    return {
        'dice': compute_dice(pred_mask, true_mask),
        'iou': compute_iou(pred_mask, true_mask),
        'hausdorff': compute_hausdorff(pred_mask, true_mask),
        'sensitivity': compute_sensitivity(pred_mask, true_mask),
        'specificity': compute_specificity(pred_mask, true_mask)
    }

# Visualization functions
def show_mask(mask, ax, random_color=False, alpha=0.6):
    """Display segmentation mask overlay."""
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([alpha])], axis=0)
    else:
        color = np.array([251/255, 252/255, 30/255, alpha])  # Yellow

    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_box(box, ax, color='blue'):
    """Draw bounding box on matplotlib axis."""
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor=color, facecolor=(0,0,0,0), lw=2))

print("✅ Evaluation metrics and helper functions defined")
print("📊 Available metrics: Dice, IoU, Hausdorff Distance, Sensitivity, Specificity")


In [ ]:
# Load SAM model (assumes already downloaded via setup_colab.ipynb)
# Following .cursor/rules: Downloads handled in setup_colab.ipynb

def load_sam_model():
    """
    Load SAM model from pretrained directory.

    Prerequisites:
    - Run setup_colab.ipynb first to download the model
    - Model should be at: pretrained/sam_vit_b_01ec64.pth or similar
    """

    # Try different SAM model variants
    sam_model_files = [
        "sam_vit_b_01ec64.pth",  # ViT-B model
        "sam_vit_l_0b3195.pth",  # ViT-L model
        "sam_vit_h_4b8939.pth",  # ViT-H model
    ]

    model_path = None
    model_type = None

    # Find available SAM model
    for filename in sam_model_files:
        potential_path = PRETRAINED_DIR / filename
        if potential_path.exists():
            model_path = potential_path
            if "vit_b" in filename:
                model_type = "vit_b"
            elif "vit_l" in filename:
                model_type = "vit_l"
            elif "vit_h" in filename:
                model_type = "vit_h"
            break

    # Check if model exists
    if model_path is None:
        print(f"❌ SAM model not found in: {PRETRAINED_DIR}")
        print("💡 Please run setup_colab.ipynb first to download the model")
        print("💡 The setup notebook will download all required models and datasets")
        print("💡 Available models: sam_vit_b_01ec64.pth, sam_vit_l_0b3195.pth, sam_vit_h_4b8939.pth")
        raise FileNotFoundError(f"SAM model not found. Run setup_colab.ipynb first.")

    try:
        print(f"🔄 Loading SAM model from: {model_path}")
        print(f"📱 Model type: {model_type}")

        # Load using SAM architecture
        sam_model = sam_model_registry[model_type](checkpoint=str(model_path))
        sam_model = sam_model.to(device)
        sam_model.eval()

        # Create predictor
        predictor = SamPredictor(sam_model)

        print(f"✅ SAM model loaded successfully")
        print(f"   Model size: {model_path.stat().st_size / (1024**3):.1f} GB")
        print(f"   Model parameters: {sum(p.numel() for p in sam_model.parameters()):,}")
        print(f"   Device: {device}")

        return sam_model, predictor

    except Exception as e:
        print(f"❌ Error loading SAM model: {e}")
        print("💡 The model file might be corrupted. Re-run setup_colab.ipynb")
        raise

# Load the SAM model
sam_model, predictor = load_sam_model()


In [ ]:
MEDICAL_DECATHLON_DIR = DATASETS_DIR / "medical_decathlon"
Task04_Hippocampus_dir = MEDICAL_DECATHLON_DIR / "Task04_Hippocampus"
Task09_Spleen_dir = MEDICAL_DECATHLON_DIR / "Task09_Spleen"
BUSI_Dataset_dir = DATASETS_DIR / "BUSI_Dataset"


In [ ]:
# Code traverses each slice of MRI volume data, keeping only slices containing hippocampus annotations to reduce unnecessary computation. Each slice is normalized to three-channel 8-bit images (SAM requires RGB image input). mri_slices list will be used for model inference, mri_slice_masks for corresponding ground truth masks.
import nibabel as nib
import numpy as np

# Get Hippocampus training set file list
imagesTr = list(Path(f"{Task04_Hippocampus_dir}/imagesTr").glob("*.nii.gz"))
labelsTr = list(Path(f"{Task04_Hippocampus_dir}/labelsTr").glob("*.nii.gz"))
image_files = sorted([f.name for f in imagesTr])
label_files = sorted([f.name for f in labelsTr])

print(f"Total {len(image_files)} MRI volumes for evaluation.")

# Prepare to store MRI dataset test slices and labels
mri_slices = []      # Will save 2D slice images (numpy arrays)
mri_slice_masks = [] # Will save corresponding 2D GT masks
mri_slice_labels = []# Save mask corresponding anatomical structure labels (1=left hippocampus, 2=right hippocampus)
mri_slice_summaries = []

for img_file, lbl_file in zip(image_files, label_files):
    img_path = f"{Task04_Hippocampus_dir}/imagesTr/{img_file}"
    lbl_path = f"{Task04_Hippocampus_dir}/labelsTr/{lbl_file}"
    # Load NIfTI volume data
    try:
      img_nii = nib.load(img_path)
      seg_nii = nib.load(lbl_path)
      img_data = img_nii.get_fdata()
      seg_data = seg_nii.get_fdata()
    except Exception as e:
      print(f"❌ Error loading NIfTI file: {e}. Skipping {img_file}")
      continue
    volume = img_data.astype(np.float32)  # 3D image data
    seg_volume = seg_data.astype(np.uint8)

    # Traverse each slice (slice direction is axis 2, i.e., axial slices)
    num_slices = img_data.shape[2]
    for z_index in range(num_slices):
        # Take certain axial slice
        slice_img = volume[:, :, z_index]
        slice_mask = seg_volume[:, :, z_index]
        # If this slice has any hippocampus structure annotation, include in evaluation
        if np.any(slice_mask > 0):
            print(f"\rimg_file:'{img_file}', lbl_file:{lbl_file}, |volume|: {volume.shape}, |seg|: {seg_volume.shape}, slice: {z_index}/{num_slices}, Slice shape: {slice_img.shape}, Mask unique labels: {np.unique(slice_mask)}", end='', flush=True)
            mri_slices.append(slice_img)
            mri_slice_masks.append(slice_mask)  # Multi-class labels (values 0,1,2)
            mri_slice_labels.append(lbl_file)  # Record the volume data file this slice belongs to
            mri_slice_summaries.append((z_index, num_slices, slice_mask.sum() / slice_img.shape[0] / slice_img.shape[1]))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.axes import Axes

def visualize_seg(fig: Figure, axs: Axes, label2box: dict, label2mask: dict, img_rgb: np.ndarray, true_mask: np.ndarray):
    # fig.clear()
    for structure_idx, (label_val, pred_mask) in enumerate(label2mask.items()):
        # Get box for this structure
        input_box = label2box[label_val]
        if input_box is None:
            continue
        x_min, y_min, x_max, y_max = input_box

        row = structure_idx
        # 1. Original image with bounding box
        ax1 = axs[row, 0]
        ax1.clear()
        ax1.imshow(img_rgb[:,:,0], cmap='gray')
        rect = plt.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min,
                            fill=False, edgecolor='red', linewidth=2)
        ax1.add_patch(rect)
        ax1.set_title(f'Input Box - Structure {label_val}')
        ax1.axis('off')

        # 2. True mask for this structure
        ax2 = axs[row, 1]
        ax2.clear()
        structure_true_mask = (true_mask == label_val).astype(np.uint8)
        ax2.imshow(img_rgb[:,:,0], cmap='gray')
        ax2.imshow(np.ma.masked_where(structure_true_mask==0, structure_true_mask),
                  cmap='spring', alpha=0.7)
        ax2.set_title(f'True Mask - Structure {label_val}')
        ax2.axis('off')

        # 3. Predicted mask for this structure
        ax3 = axs[row, 2]
        ax3.clear()
        ax3.imshow(img_rgb[:,:,0], cmap='gray')
        ax3.imshow(np.ma.masked_where(pred_mask==0, pred_mask),
                  cmap='autumn', alpha=0.7)
        ax3.set_title(f'Pred Mask - Structure {label_val}')
        ax3.axis('off')

        # 4. Overlay comparison
        ax4 = axs[row, 3]
        ax4.clear()
        ax4.imshow(img_rgb[:,:,0], cmap='gray')
        # True mask in green, pred mask in red
        ax4.imshow(np.ma.masked_where(structure_true_mask==0, structure_true_mask),
                  cmap='Greens', alpha=0.5)
        ax4.imshow(np.ma.masked_where(pred_mask==0, pred_mask),
                  cmap='Reds', alpha=0.5)
        ax4.set_title(f'Overlay - Structure {label_val}')
        ax4.axis('off')

    # Combined visualization (third row)
    # 1. All bounding boxes
    ax_combined1 = axs[2, 0]
    ax_combined1.clear()
    ax_combined1.imshow(img_rgb[:,:,0], cmap='gray')
    colors = ['red', 'blue']
    for i, (label_val, input_box) in enumerate(label2box.items()):
        x_min, y_min, x_max, y_max = input_box
        rect = plt.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min,
                            fill=False, edgecolor=colors[i % len(colors)], linewidth=2)
        ax_combined1.add_patch(rect)
    ax_combined1.set_title('All Input Boxes')
    ax_combined1.axis('off')

    # 2. Combined true mask
    ax_combined2 = axs[2, 1]
    ax_combined2.clear()
    ax_combined2.imshow(img_rgb[:,:,0], cmap='gray')
    ax_combined2.imshow(np.ma.masked_where((true_mask > 0)==0, (true_mask > 0)),
                        cmap='spring', alpha=0.7)
    ax_combined2.set_title('Combined True Mask')
    ax_combined2.axis('off')

    # 3. Combined predicted mask
    ax_combined3 = axs[2, 2]
    ax_combined3.clear()
    ax_combined3.imshow(img_rgb[:,:,0], cmap='gray')
    ax_combined3.imshow(np.ma.masked_where(pred_mask_total==0, pred_mask_total),
                        cmap='autumn', alpha=0.7)
    ax_combined3.set_title('Combined Pred Mask')
    ax_combined3.axis('off')

    # 4. Final comparison
    ax_combined4 = axs[2, 3]
    ax_combined4.clear()
    ax_combined4.imshow(img_rgb[:,:,0], cmap='gray')
    ax_combined4.imshow(np.ma.masked_where((true_mask > 0)==0, (true_mask > 0)),
                        cmap='Greens', alpha=0.5)
    ax_combined4.imshow(np.ma.masked_where(pred_mask_total==0, pred_mask_total),
                        cmap='Reds', alpha=0.5)
    ax_combined4.set_title('Final Overlay')
    ax_combined4.axis('off')

    fig.canvas.draw()
    fig.canvas.flush_events()

all_labels = [1, 2]

def sam_segment(slice_img: np.ndarray, slice_mask: np.ndarray, predictor: SamPredictor):
    """
    Perform segmentation using SAM predictor on a 2D slice.

    Args:
        slice_img: Input image slice
        slice_mask: Ground truth mask
        predictor: SAM predictor instance

    Returns:
        tuple: (label2mask, label2box, pred_mask_total, true_mask, img_rgb)
    """
    # Normalize and prepare image
    mn, mx = slice_img.min(), slice_img.max()
    slice_img_norm = ((slice_img - mn) / (mx - mn + 1e-8) * 255.0).astype(np.uint8)
    slice_img_resized = cv2.resize(slice_img_norm, (1024, 1024), interpolation=cv2.INTER_CUBIC)
    img_rgb = np.stack([slice_img_resized]*3, axis=-1)

    true_mask = cv2.resize(slice_mask.astype(np.uint8)*127, (1024, 1024), interpolation=cv2.INTER_NEAREST)
    true_mask = true_mask.astype(np.uint8) // 127

    predictor.set_image(img_rgb)  # Set image for SAM predictor
    pred_mask_total = np.zeros(true_mask.shape, dtype=bool)

    # Predict each structure separately (labels 1 and 2 correspond to two hippocampi)
    label2mask = {}
    label2box = {}
    for structure_idx, label_val in enumerate(all_labels):
        # Skip structures that don't exist
        if np.sum(true_mask == label_val) == 0:
            continue

        # Calculate bounding box for this structure
        ys, xs = np.where(true_mask == label_val)
        y_min, y_max = ys.min(), ys.max()
        x_min, x_max = xs.min(), xs.max()
        input_box = np.array([x_min, y_min, x_max, y_max])
        label2box[label_val] = input_box

        # Use bounding box prompt for prediction
        masks, scores, _ = predictor.predict(box=input_box[None, :], point_coords=None, point_labels=None, multimask_output=False)
        pred_mask = masks[0]  # Output mask
        label2mask[label_val] = pred_mask

        # Add this structure's predicted mask to total mask
        pred_mask_total = np.logical_or(pred_mask_total, pred_mask)

    return label2mask, label2box, pred_mask_total, true_mask, img_rgb


In [ ]:
dice_list_hc = []
iou_list_hc = []
hd_list_hc = []

good_slices = []

for idx, (slice_img, slice_mask, slice_label, slice_info) in enumerate(zip(mri_slices, mri_slice_masks, mri_slice_labels, mri_slice_summaries)):
    # if idx > 20:
    #    break
    label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
    combined_true_mask = (true_mask > 0)

    if slice_info[2] > 0.12 and len(label2box) > 1:
        good_slices.append(idx)

    # Calculate evaluation metrics
    dice = compute_dice(pred_mask_total, combined_true_mask)
    iou = compute_iou(pred_mask_total, combined_true_mask)
    hd = compute_hausdorff(pred_mask_total, combined_true_mask)
    dice_list_hc.append(dice)
    iou_list_hc.append(iou)
    hd_list_hc.append(hd)
    # Print processing progress
    print(f"\rProcessing slice {idx+1}/{len(mri_slices)}: {slice_label}, "
          f"Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px", end='', flush=True)


In [ ]:
dice_list_hc = np.array(dice_list_hc)
iou_list_hc = np.array(iou_list_hc)
hd_list_hc = np.array([d for d in hd_list_hc if math.isfinite(d)])

# Calculate average metrics
dice_mean_hc = np.mean(dice_list_hc)
iou_mean_hc = np.mean(iou_list_hc)
hd_mean_hc = np.mean(hd_list_hc)

print(f"\n\n📊 Final Results Summary:")
print(f"   Total slices processed: {len(mri_slices)}")
print(f"   Slices with DICE < 0.5: {sum(1 for d in dice_list_hc if d < 0.5)}")
print(f"   Hippocampus MRI dataset: Average Dice = {dice_mean_hc:.4f}, Average IoU = {iou_mean_hc:.4f}, Average Hausdorff distance = {hd_mean_hc:.2f} pixel")

good_dice_mean_hc = np.mean(dice_list_hc[good_slices])
good_iou_mean_hc = np.mean(iou_list_hc[good_slices])
good_hd_mean_hc = np.mean(hd_list_hc[good_slices])
print(f"   In slices with maskedRatio > 0.12 and both labels existing:")
print(f"   Average Dice = {good_dice_mean_hc:.4f}, Average IoU = {good_iou_mean_hc:.4f}, Average Hausdorff distance = {good_hd_mean_hc:.2f} pixel")


In [ ]:
# Clean up and visualize hippocampus results
fig = None
plt.ioff()  # Turn off interactive mode
# Set up visualization
plt.ion()  # Turn on interactive mode
fig, axs = plt.subplots(3, 4)

idx = good_slices[np.random.randint(0, len(good_slices))]
slice_img, slice_mask, slice_label, slice_info = mri_slices[idx], mri_slice_masks[idx], mri_slice_labels[idx], mri_slice_summaries[idx]
label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
# Visualize each structure separately
visualize_seg(fig, axs, label2box, label2mask, img_rgb, true_mask)

# Update figure title with metrics - highlight poor performance
dice, iou, hd = dice_list_hc[idx], iou_list_hc[idx], hd_list_hc[idx]
fig.suptitle(f'Slice {idx+1}/{len(mri_slices)}\n'
            f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
            fontsize=14, y=0.95, color='red')
# Clean up
plt.ioff()  # Turn off interactive mode
plt.tight_layout()
plt.show()
plt.close(fig)
fig = None


In [ ]:
# Get Spleen dataset file list
ct_image_files = list(Path(f"{Task09_Spleen_dir}/imagesTr").glob("*.nii.gz"))
ct_label_files = list(Path(f"{Task09_Spleen_dir}/labelsTr").glob("*.nii.gz"))
ct_image_files = sorted([f.name for f in ct_image_files])
ct_label_files = sorted([f.name for f in ct_label_files])
print(f"Total {len(ct_image_files)} CT volumes for evaluation.")

ct_slices = []
ct_slice_masks = []

for img_file, lbl_file in zip(ct_image_files, ct_label_files):
    print(f"\rimg_file:'{img_file}', lbl_file:{lbl_file}", end='', flush=True)
    img_path = f"{Task09_Spleen_dir}/imagesTr/{img_file}"
    lbl_path = f"{Task09_Spleen_dir}/labelsTr/{lbl_file}"
    img_nii = nib.load(img_path)
    lbl_nii = nib.load(lbl_path)
    img_data = img_nii.get_fdata().astype(np.float32)
    lbl_data = lbl_nii.get_fdata().astype(np.uint8)
    # Traverse axial slices
    num_slices = img_data.shape[0]
    for k in range(num_slices):
        slice_img = img_data[k, :, :]
        slice_lbl = lbl_data[k, :, :]
        if np.any(slice_lbl == 1):  # If this slice contains spleen
            # Clip CT slice gray values to [-1000, 1000] HU range and normalize to 0-255
            slice_clip = np.clip(slice_img, -1000, 1000)
            ct_slices.append(slice_clip)
            ct_slice_masks.append(slice_lbl)  # Binary mask (0 background, 1 spleen)


In [ ]:
dice_list_spleen = []
iou_list_spleen = []
hd_list_spleen = []

for idx, (slice_img, slice_mask) in enumerate(zip(ct_slices, ct_slice_masks)):
    # if idx > 20:
    #    break
    label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
    combined_true_mask = (true_mask == 1)

    # Calculate evaluation metrics
    dice = compute_dice(pred_mask_total, combined_true_mask)
    iou = compute_iou(pred_mask_total, combined_true_mask)
    hd = compute_hausdorff(pred_mask_total, combined_true_mask)
    dice_list_spleen.append(dice)
    iou_list_spleen.append(iou)
    hd_list_spleen.append(hd)
    # Print processing progress
    print(f"\rProcessing CT slice {idx+1}/{len(ct_slices)}: "
          f"Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px", end='', flush=True)


In [ ]:
# Calculate average metrics for spleen
dice_mean_spleen = np.mean(dice_list_spleen)
iou_mean_spleen = np.mean(iou_list_spleen)
hd_mean_spleen = np.mean([d for d in hd_list_spleen if math.isfinite(d)])
print(f"\n\n📊 Final Results Summary:")
print(f"   Total slices processed: {len(ct_slices)}")
print(f"   Slices with DICE < 0.5: {sum(1 for d in dice_list_spleen if d < 0.5)}")
print(f"Spleen CT dataset: Average Dice = {dice_mean_spleen:.4f}, Average IoU = {iou_mean_spleen:.4f}, Average Hausdorff distance = {hd_mean_spleen:.2f} pixel")


In [ ]:
# Visualize spleen results
plt.ion()  # Turn on interactive mode
fig, axs = plt.subplots(3, 4)

idx = np.random.randint(0, len(ct_slices))
slice_img, slice_mask = ct_slices[idx], ct_slice_masks[idx]
label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
visualize_seg(fig, axs, label2box, label2mask, img_rgb, true_mask)

# Update figure title with metrics - highlight poor performance
dice, iou, hd = dice_list_spleen[idx], iou_list_spleen[idx], hd_list_spleen[idx]
fig.suptitle(f'Slice {idx+1}/{len(ct_slices)}\n'
            f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
            fontsize=14, y=0.95, color='red')
# Clean up
plt.ioff()  # Turn off interactive mode
plt.tight_layout()
plt.show()
plt.close(fig)
fig = None


In [ ]:
import cv2
import os
import glob

ultrasound_images = []
ultrasound_masks = []

# Process benign and malignant folders
for cls in ["benign", "malignant"]:
    image_paths = glob.glob(f"{BUSI_Dataset_dir}/{cls}/*.png")
    for img_path in image_paths:
        if "_mask" in img_path:
            continue  # Skip mask files
        # Read ultrasound image (grayscale PNG, cv2.imread reads as BGR three-channel by default)
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        # Create empty mask with same dimensions as image
        mask_total = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
        # Image file name base part (remove path and extension)
        base_name = os.path.splitext(img_path)[0]
        # Merge all mask files for this image (may have multiple tumors)
        mask_idx = 1
        while True:
            mask_path = f"{base_name}_mask.png" if mask_idx == 1 else f"{base_name}_mask_{mask_idx}.png"
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask is not None:
                    mask_binary = (mask > 127).astype(np.uint8)
                    mask_total = np.logical_or(mask_total, mask_binary).astype(np.uint8)
                mask_idx += 1
            else:
                break
        # If this image has tumor annotation, save it
        if mask_total.sum() > 0:
            ultrasound_images.append(img_rgb.mean(axis=2).astype(np.float32))
            ultrasound_masks.append(mask_total)

print(f"Breast ultrasound images total: {len(ultrasound_images)} (benign+malignant), masks total: {len(ultrasound_masks)}")


In [ ]:
dice_list_us = []
iou_list_us = []
hd_list_us = []

for idx, (slice_img, slice_mask) in enumerate(zip(ultrasound_images, ultrasound_masks)):
    # if idx > 20:
    #    break
    label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
    combined_true_mask = (true_mask == 1)

    # Calculate evaluation metrics
    dice = compute_dice(pred_mask_total, combined_true_mask)
    iou = compute_iou(pred_mask_total, combined_true_mask)
    hd = compute_hausdorff(pred_mask_total, combined_true_mask)
    dice_list_us.append(dice)
    iou_list_us.append(iou)
    hd_list_us.append(hd)
    # Print processing progress
    print(f"\rProcessing ultrasound slice {idx+1}/{len(ultrasound_images)}: "
          f"Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px", end='', flush=True)


In [ ]:
# Calculate average metrics for ultrasound
dice_mean_us = np.mean(dice_list_us)
iou_mean_us = np.mean(iou_list_us)
hd_mean_us = np.mean([d for d in hd_list_us if math.isfinite(d)])
print(f"\n\n📊 Final Results Summary:")
print(f"   Total slices processed: {len(ultrasound_images)}")
print(f"   Slices with DICE < 0.5: {sum(1 for d in dice_list_us if d < 0.5)}")
print(f"Ultrasound dataset: Average Dice = {dice_mean_us:.4f}, Average IoU = {iou_mean_us:.4f}, Average Hausdorff distance = {hd_mean_us:.2f} pixel")


In [ ]:
# Visualize ultrasound results
if fig is not None:
    fig.clear()
    plt.close(fig)
    fig = None
plt.ion()  # Turn on interactive mode
fig, axs = plt.subplots(3, 4)

idx = np.random.randint(0, len(ultrasound_images))
slice_img, slice_mask = ultrasound_images[idx], ultrasound_masks[idx]
label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
visualize_seg(fig, axs, label2box, label2mask, img_rgb, true_mask)

# Update figure title with metrics - highlight poor performance
dice, iou, hd = dice_list_us[idx], iou_list_us[idx], hd_list_us[idx]
fig.suptitle(f'Slice {idx+1}/{len(ultrasound_images)}\n'
            f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
            fontsize=14, y=0.95, color='red')
# Clean up
plt.ioff()  # Turn off interactive mode
plt.tight_layout()
plt.show()
plt.close(fig)
fig = None


In [ ]:
# Final comprehensive results summary
print("\n🏆 SAM Baseline Test Results Summary:")
print("=" * 60)

print(f"📊 Hippocampus MRI Dataset:")
print(f"   Mean Dice: {dice_mean_hc:.4f}")
print(f"   Mean IoU: {iou_mean_hc:.4f}")
print(f"   Mean Hausdorff: {hd_mean_hc:.2f}px")
print(f"   Total slices: {len(mri_slices)}")

print(f"\n📊 Spleen CT Dataset:")
print(f"   Mean Dice: {dice_mean_spleen:.4f}")
print(f"   Mean IoU: {iou_mean_spleen:.4f}")
print(f"   Mean Hausdorff: {hd_mean_spleen:.2f}px")
print(f"   Total slices: {len(ct_slices)}")

print(f"\n📊 Breast Ultrasound Dataset:")
print(f"   Mean Dice: {dice_mean_us:.4f}")
print(f"   Mean IoU: {iou_mean_us:.4f}")
print(f"   Mean Hausdorff: {hd_mean_us:.2f}px")
print(f"   Total images: {len(ultrasound_images)}")

print(f"\n🎯 Overall Summary:")
all_dice = dice_list_hc + dice_list_spleen + dice_list_us
all_iou = iou_list_hc + iou_list_spleen + iou_list_us
all_hd = [d for d in (hd_list_hc + hd_list_spleen + hd_list_us) if math.isfinite(d)]

print(f"   Average across all datasets:")
print(f"   Mean Dice: {np.mean(all_dice):.4f}")
print(f"   Mean IoU: {np.mean(all_iou):.4f}")
print(f"   Mean Hausdorff: {np.mean(all_hd):.2f}px")
print(f"   Total samples: {len(all_dice)}")

print(f"\n💡 Notes:")
print(f"   - This baseline uses original SAM model with bounding box prompts")
print(f"   - SAM is a general vision model, not specifically trained for medical imaging")
print(f"   - Results show capability on diverse medical imaging modalities")
print(f"   - For comparison with MedSAM which is fine-tuned for medical images")

print(f"\n✅ SAM baseline evaluation completed successfully!")
